In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Constants and Parameters based on VMD and Soft Pomeron models [cite: 42, 43, 56]
ALPHA_EM = 1/137.036
PROTON_MASS = 0.93827  # GeV
F0_MASS = 0.980        # GeV (Standard f0(980) mass)

class MesonProductionModel:
    def __init__(self, meson_mass=F0_MASS):
        self.m_x = meson_mass
        # Phenomenological parameters for diffractive scattering [cite: 43, 467]
        self.s0 = 1.0  # GeV^2 (energy scale)
        self.alpha_p_prime = 0.25  # GeV^-2 (Pomeron slope)
        self.epsilon = 0.08        # Pomeron intercept minus 1

    def mandelstam_t_min(self, s):
        """Calculates minimum momentum transfer t_min[cite: 36, 459]."""
        e_cm = np.sqrt(s)
        p_initial = np.sqrt((s - PROTON_MASS**2)**2) / (2 * e_cm)
        p_final = np.sqrt((s - (PROTON_MASS + self.m_x)**2) * (s - (PROTON_MASS - self.m_x)**2)) / (2 * e_cm)
        t_min = self.m_x**4 / (4 * s) - (p_initial - p_final)**2
        return t_min

    def photoproduction_cross_section(self, s):
        """
        Models photoproduction cross section sigma(gamma p -> f0 p) 
        using a Pomeron-based diffractive model[cite: 43, 463].
        """
        if s < (PROTON_MASS + self.m_x)**2:
            return 0.0
        
        # Energy dependence from soft Pomeron model 
        # sigma ~ (s/s0)^(2*epsilon)
        energy_factor = (s / self.s0)**(2 * self.epsilon)
        
        # Typical diffractive cross section scale (phenomenological)
        sigma_base = 0.5 # microbarns (placeholder for f0 scale)
        return sigma_base * energy_factor

    def electroproduction_cross_section(self, s, Q2):
        """
        Models electroproduction cross section sigma(e p -> e f0 p) 
        incorporating virtual photon flux.
        """
        # VMD propagator for virtual photon 
        vmd_factor = (1 / (1 + Q2 / (F0_MASS**2)))**2
        
        photo_sigma = self.photoproduction_cross_section(s)
        return photo_sigma * vmd_factor

# --- Simulation and Visualization ---
s_range = np.linspace(5, 100, 100)  # Center of mass energy squared
model = MesonProductionModel()

# Generate photo-production data
photo_data = [model.photoproduction_cross_section(s) for s in s_range]

# Generate electro-production data for different Q^2 (virtuality) values
q2_values = [0.1, 1.0, 5.0]
electro_results = {q2: [model.electroproduction_cross_section(s, q2) for s in s_range] for q2 in q2_values}

# Plotting the model predictions
plt.figure(figsize=(10, 6))
plt.plot(np.sqrt(s_range), photo_data, label='Photoproduction ($Q^2=0$)', linewidth=2)

for q2, data in electro_results.items():
    plt.plot(np.sqrt(s_range), data, '--', label=f'Electroproduction ($Q^2={q2}$ GeV$^2$)')

plt.xlabel('$\sqrt{s}$ (GeV)')
plt.ylabel('$\sigma$ ($\mu b$)')
plt.title('Predicted $f_0$ Meson Production Cross Sections')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()